# Week 4 — Which Things Occur Together?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/advanced-data-mining-course/blob/main/weeks/04/Week_04_Association_Rules_Beyond_Popularity.ipynb)

**Driving question:** Does “bought together” tell us more than product popularity?

Week 3 ranked unusual customer histories for review. Today we investigate recurring product combinations. The connection is the baseline: an unusual observation needs a reference population, and a seemingly impressive association needs a comparison with what was common already. We use the same retail extract, rebuilding the analysis independently.

By the end, you should be able to define a transaction and an item; hand-calculate support, confidence, and lift; explain why frequent-pattern search can prune some combinations; use frequent pattern growth (FP-growth) to find bounded product combinations; and justify a small rule shortlist with original counts, a counterexample, and a limitation. An association rule is not a prediction of purchase order or proof that one purchase causes another.

Opening puzzle: “Every buyer of product A also buys product B.” Would you recommend A to increase B sales? Write what else you would need to know. In particular, what if almost everyone already buys B?

The provisional core is 180 minutes excluding breaks: opening/setup 10, Reading A 15, hand counts 15, Reading B 15, basket construction 15, Investigation 1 40, Reading C 15, Investigation 2 40, synthesis 15. Reading windows include reflection and written checkpoints. Choose one route per investigation; the other is an extension. All activities here are formative. No assignment is released by this notebook.

## Setup — one notebook, automatically prepared data

The Colab button targets the published student repository. Until this lecture is published there, use the local setup: `uv sync --locked`, then `uv run jupyter lab` from the course root. No variables, exports, or saved work from earlier lectures are needed.

In Colab, connect to a CPU runtime and choose **Runtime → Run all**. Dependencies install automatically; the public retail file downloads, is checked against its checksum, and is cached. No manual upload, ZIP extraction, private account, or paid service is required. Restart and run all if Colab requests a runtime restart after installation. Locally, the bundled data are the fallback when internet access is unavailable. First-time installation or downloading needs internet; external readings are optional, not runtime dependencies.

Run the defaults before changing settings. Record a prediction, change one setting, and rerun its cell and the dependent analysis cells. Restart and run all if you are unsure which state is current. The exercises modify working examples rather than asking you to build data-preparation infrastructure.

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
        'numpy>=2,<3','pandas>=2.2,<3','matplotlib>=3.9,<4',
        'scikit-learn>=1.6,<1.8','mlxtend==0.23.4'])
    print('Colab dependencies ready. Continue, or restart and run all if requested.')
else:
    print('Local runtime: using the locked course environment.')

In [ ]:
from pathlib import Path
from itertools import combinations
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlxtend.frequent_patterns import fpgrowth, association_rules, apriori
import mlxtend

pd.set_option('display.max_columns',12)
plt.rcParams.update({'figure.figsize':(8,4),'font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False})
print('NumPy:',np.__version__,'| pandas:',pd.__version__,'| mlxtend:',mlxtend.__version__)

In [ ]:
# Prefer the bundled/local cache; otherwise fetch the public course dataset.
import urllib.request
import urllib.error

filename = 'online_retail_teaching.csv.gz'
DATA_URL = 'https://raw.githubusercontent.com/rrfhwn/advanced-data-mining-course/main/data/online_retail_teaching.csv.gz'
expected_sha256 = '8503a734cd203ed63df3292c2a12d0b803206dc18ed67a7dc83664fd307d96d8'
candidates = [Path.cwd() / filename]
for folder in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(folder / 'data' / filename)
data_path = next((p for p in candidates if p.is_file()), None)

if data_path is None:
    data_path = Path.cwd() / 'data' / filename
    print('Downloading the course dataset (approximately 610 KB)...')
    try:
        with urllib.request.urlopen(DATA_URL, timeout=60) as response:
            payload = response.read()
    except (urllib.error.URLError, TimeoutError) as error:
        raise RuntimeError(
            'Could not download the course dataset. Check your internet connection and retry. '
            'If the URL is unavailable, ask the instructor to check the public course release. '
            f'URL: {DATA_URL}'
        ) from error
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise ValueError('Downloaded data does not match this notebook version. Reopen the current course notebook.')
    data_path.parent.mkdir(parents=True, exist_ok=True)
    data_path.write_bytes(payload)
else:
    print('Using the cached course dataset.')

if hashlib.sha256(data_path.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError('Cached dataset differs from this notebook version. Use a fresh Colab runtime or restore the matching course data locally.')
raw = pd.read_csv(data_path, dtype={'InvoiceNo': 'string', 'StockCode': 'string',
                                  'CustomerID': 'string'}, parse_dates=['InvoiceDate'])
assert len(raw) > 0 and raw.InvoiceDate.notna().all()
print(f'{len(raw):,} rows; {raw.CustomerID.nunique()} observed customer IDs')
display(raw.head(4))

## Reading A — A rule is a comparison of counts

**Read independently for 10–15 minutes, including the checkpoint.** Explain the denominator of each quantity before deciding whether a rule is interesting.

### First decide what belongs together

A transaction is the observation within which items can co-occur. In a market-basket analysis it might be one invoice, one shopping visit, or all purchases by a customer during a defined period. These are not interchangeable. Two products bought months apart can co-occur in a customer-history basket while never appearing on one invoice. The method will calculate either definition correctly; the interpretation is our responsibility.

An item is a distinct thing whose presence we record. Today it is a product stock code, not a description string or a quantity. A basket containing five units of the same product records that product as present once. This is binary presence: yes or no. Repeating a line or increasing its quantity does not give the basket extra votes. Quantity-aware questions require another representation rather than silently treating units as independent transactions.

An itemset is a set of products, such as {milk, bread}. Sets have no purchase order and no repeated members. A rule writes one nonempty itemset on the left, the antecedent, and a disjoint nonempty itemset on the right, the consequent. The rule {milk} → {bread} reads: among transactions containing milk, how often is bread also present? “Antecedent” means the conditioning set here, not an earlier event. The arrow does not establish a temporal sequence or an intervention.

### Work through eight baskets

Imagine eight deliberately constructed baskets. Three contain milk and bread, one contains milk without bread, three contain bread without milk, and one contains neither. Milk therefore appears in four baskets and bread in six. The joint count is three. Keep all eight baskets in the population, including the last basket: an absence of these two products is part of the evidence about their prevalence.

Let $N$ be the total number of baskets, $n_A$ the count containing every item in antecedent $A$, $n_B$ the count containing every item in consequent $B$, and $n_{AB}$ the count containing all items in their union. Our probabilities are observed proportions in this specified extract, not known population probabilities. Three common measures are

$$\operatorname{support}(A\to B)=n_{AB}/N,$$
$$\operatorname{confidence}(A\to B)=n_{AB}/n_A,$$
$$\operatorname{lift}(A\to B)=\frac{n_{AB}/n_A}{n_B/N}.$$

Support is the fraction of all baskets containing the combination; some libraries also call the integer joint count “support count,” so name which you report. Confidence is the conditional fraction containing B among A baskets. Lift divides that conditional fraction by B's overall prevalence. Support and confidence lie between zero and one when defined. Lift is nonnegative and can exceed one; it has no common fixed upper bound across datasets.

For milk → bread, support is 3/8 = 0.375 and confidence is 3/4 = 0.75. Bread's baseline prevalence is 6/8 = 0.75, so lift is one. The apparently attractive 75% confidence adds nothing to that empirical baseline. Reversing the arrow changes confidence to 3/6 = 0.5. Joint support and lift stay the same for a fixed pair and population, while the conditioning question changes.

### What a high value does not establish

Despite its name, confidence here is a conditional fraction, not a confidence interval or a statistical uncertainty level.

Lift above one means the combination occurs more often than the product of its marginal prevalences would suggest. It does not establish causation, statistical significance, or future performance. A promotion, product family, purchasing context, or a few repeated buyers may help explain the pattern. Lift below one indicates less co-occurrence than that baseline; it does not prove products substitute for one another. Lift one describes this empirical comparison, not a universal guarantee of independence.

The denominators also have boundary cases. If no baskets contain A, confidence is undefined. If B is absent, lift is undefined. A zero joint count with positive A and B counts gives support, confidence, and lift zero. If the population itself is empty, proportions are undefined. We use missing numerical values for undefined measures rather than replacing them with zeros or infinity. The helper below makes these cases inspectable, including an item absent from the supplied columns.

### Checkpoint before running

Calculate milk → bread and bread → milk. Which values change? If we removed the basket containing neither, would confidence change? Would lift? Write why “75% confidence” is incomplete without a baseline.

<details><summary>Self-check — open after writing</summary>

The forward and reverse confidences are 0.75 and 0.5; both have support 0.375 and lift 1. Removing the neither basket changes N from eight to seven, leaving conditional confidence unchanged but changing support and lift. Bread's baseline becomes 6/7, so forward lift becomes 0.875. The population definition changes the reference even when A and B counts stay fixed.

</details>

In [ ]:
def rule_evidence(presence, antecedent, consequent):
    A, B = frozenset(antecedent), frozenset(consequent)
    if not A or not B or A & B:
        raise ValueError('Use nonempty disjoint antecedent and consequent sets.')
    if not presence.dtypes.eq(bool).all():
        raise ValueError('Use a boolean presence table, not quantities or missing values.')
    def contains(items):
        if not items.issubset(presence.columns):
            return pd.Series(False,index=presence.index)
        return presence[sorted(items)].all(axis=1)
    in_A, in_B = contains(A), contains(B)
    N, n_A, n_B = len(presence),int(in_A.sum()),int(in_B.sum())
    n_AB = int((in_A & in_B).sum())
    support = n_AB/N if N else np.nan
    confidence = n_AB/n_A if n_A else np.nan
    baseline = n_B/N if N else np.nan
    lift = confidence/baseline if n_A and n_B and N else np.nan
    return pd.Series({'baskets':N,'A_count':n_A,'B_count':n_B,'joint_count':n_AB,
        'support':support,'confidence':confidence,'B_baseline':baseline,'lift':lift})

toy_sets = [{'milk','bread'},{'milk','bread'},{'milk','bread'},{'milk'},
            {'bread'},{'bread'},{'bread'},{'apples'}]
toy = pd.DataFrame([{item:item in basket for item in ['milk','bread','apples']}
                    for basket in toy_sets],index=[f'T{i+1}' for i in range(8)])
display(toy)
display(pd.DataFrame({'milk → bread':rule_evidence(toy,{'milk'},{'bread'}),
                     'bread → milk':rule_evidence(toy,{'bread'},{'milk'})}).round(3))
# TODO 1: Predict, then exclude T8. Compare denominators before interpreting lift.
drop_neither = False
toy_view = toy.drop(index='T8') if drop_neither else toy
display(rule_evidence(toy_view,{'milk'},{'bread'}).round(3))

In [ ]:
# Four disjoint cells: these are basket counts, not model predictions.
contingency = pd.DataFrame([[3,1],[3,1]],index=['Milk present','Milk absent'],
                          columns=['Bread present','Bread absent'])
fig,ax = plt.subplots(figsize=(6,3.4))
image = ax.imshow(contingency,cmap='Blues',vmin=0,vmax=4)
ax.set(xticks=[0,1],xticklabels=contingency.columns,yticks=[0,1],yticklabels=contingency.index,
       title='Eight constructed baskets — count every cell once')
for row in range(2):
    for col in range(2): ax.text(col,row,str(contingency.iloc[row,col]),ha='center',va='center')
fig.colorbar(image,ax=ax,label='Basket count'); plt.tight_layout(); plt.show()
# TODO 2: Explain confidence using one row, then B's prevalence using both rows.

## Reading B — From recorded lines to a bounded pattern search

**Read and reflect for 10–15 minutes.** Your goal is to explain why the basket definition and search boundary are part of the result, not merely setup work.

### The same extract, a different observation unit

The file is an attributed teaching adaptation of UCI Online Retail under Creative Commons Attribution 4.0. See the [data provenance, field guide, and selection limitations](../../data/README.md). The raw file is unchanged; today's basket preparation happens in a separate table.

Earlier lectures constructed customer recency, frequency, and monetary summaries. We now return to invoice lines and group products within invoices. One identified customer can contribute many invoices; those observations are not automatically independent. The supplied extract selected customer IDs and kept their observed histories, so frequent purchasers contribute more baskets. This is useful for investigating this extract, not estimating the prevalence of combinations across the retailer or all shoppers.

We use identified, positive, noncancelled purchases and preserve the raw source table. Excluding unknown IDs maintains a consistent cohort for the later invoice-versus-customer comparison; invoice construction could include anonymous purchases in another study if its coverage and selection mechanism were stated. Excluding cancellations defines a positive-purchase basket; it does not reconstruct net purchases after returns. Week 3's large cancellation example is a reminder that this representation is not an accounting ledger.

We retain stock codes following five digits with an optional trailing capital letter. This transparent syntax rule excludes many service and adjustment codes, but it is only a product-code heuristic, not verified product classification. A new dataset could use a different scheme. Descriptions are display labels only: two differently described records with the same code still represent one item here. Inspect the labels when explaining a rule, but do not merge codes merely because their names look similar.

Within an invoice, repeated product lines and quantities greater than one become a single presence. Do not drop a single-product invoice just because it cannot contain a pair: it contributes to the denominator and to product prevalence. Similarly, after selecting a small catalog, retain invoices that contain none of those selected products. An all-false row means “none of the selected catalog,” not an invoice with no actual purchases. Removing it silently changes the population.

### Why the catalog is small

We rank products by the number of distinct eligible invoices containing them, breaking ties by code, then retain the top 40. Every eligible invoice remains a row. This restricts which rules can be discovered; it does not redefine the basket population. The selected catalog comes from this exploratory dataset. It is not a vocabulary fitted on a training subset for a claimed future evaluation. Week 5 will separate discovery choices from held-out evidence.

With 40 products there are 780 distinct unordered pairs and up to 1,560 directed pair rules. Considering arbitrary-length itemsets would create far more combinations: even a modest catalog has exponentially many subsets. A procedure that looks fast on our bounded teaching case is not automatically scalable to an unbounded catalog. We limit mining to itemsets of length at most two and require at least 12 joint occurrences. The exact fractional support threshold is derived from this count and the number of baskets, not guessed independently of dataset size.

### Pruning without losing qualifying patterns

Apriori uses a useful property: any superset can occur no more frequently than each of its subsets. If {milk, bread} occurs in three baskets, {milk, bread, eggs} cannot occur in four. When an itemset fails a minimum-support threshold, all larger sets containing it can be pruned. For a proposed triple, an infrequent pair is enough to reject it without counting the triple. This property applies to support under the same population, not to confidence or lift.

Frequent pattern growth, abbreviated FP-growth, uses the same frequency objective but a different search organization. It orders frequent items, compresses repeated transaction prefixes into a tree, and searches conditional pattern structures instead of generating the same style of candidate lists as Apriori. We use mlxtend's established implementation rather than writing a replacement mining algorithm. The small hand-counted example remains our independent check of what the measures mean.

Finding frequent itemsets and generating rules are separate steps. First find item combinations meeting the support requirement. Then split a combination into antecedent and consequent, calculate its measures, and apply any desired filtering. Our pair-only core yields singleton-to-singleton rules in both directions. A minimum confidence filter is a selection preference, not a guarantee of usefulness. We initially retain all generated directions so that high-confidence and high-lift shortlists can be compared explicitly.

### Checkpoint

Explain why a top-catalog-empty invoice stays in the table. For a triple containing an infrequent pair, explain Apriori's pruning decision. Name two search limits and one important pattern they might exclude.

<details><summary>Self-check</summary>

The invoice belongs to the fixed eligible population even if it contains no selected product. Support cannot increase when items are added to an itemset, so an infrequent pair rules out every containing triple at the same threshold. A top-40 catalog, pair-length limit, and 12-occurrence minimum bound work but omit rare products, longer combinations, and lower-count patterns. These limits define coverage, not truth.

</details>

In [ ]:
eligible = raw.drop_duplicates().copy()
eligible = eligible[(~eligible.InvoiceNo.str.upper().str.startswith('C',na=False)) &
    eligible.Quantity.gt(0) & eligible.UnitPrice.gt(0) & eligible.CustomerID.notna() &
    eligible.StockCode.str.fullmatch(r'\d{5}[A-Z]?',na=False)].copy()
invoice_ids = pd.Index(sorted(eligible.InvoiceNo.unique()),name='InvoiceNo')
customer_ids = pd.Index(sorted(eligible.CustomerID.unique()),name='CustomerID')
invoice_items = eligible[['InvoiceNo','StockCode']].drop_duplicates()
popularity = invoice_items.StockCode.value_counts().rename_axis('StockCode').reset_index(name='invoice_count')
popularity = popularity.sort_values(['invoice_count','StockCode'],ascending=[False,True])
catalog = list(popularity.head(40).StockCode)
description_counts = (eligible.dropna(subset=['Description'])
    .groupby(['StockCode','Description']).size().rename('rows').reset_index()
    .sort_values(['StockCode','rows','Description'],ascending=[True,False,True]))
labels = description_counts.drop_duplicates('StockCode').set_index('StockCode').Description.to_dict()

def make_presence(lines, unit, vocabulary, universe):
    if len(set(vocabulary)) != len(vocabulary) or not universe.is_unique:
        raise ValueError('Catalog and basket IDs must be unique.')
    pairs = lines.loc[lines.StockCode.isin(vocabulary),[unit,'StockCode']].drop_duplicates()
    if pairs.empty:
        return pd.DataFrame(False,index=universe,columns=vocabulary,dtype=bool)
    table = pd.crosstab(pairs[unit],pairs.StockCode)
    return table.reindex(index=universe,columns=vocabulary,fill_value=0).gt(0)

baskets = make_presence(eligible,'InvoiceNo',catalog,invoice_ids)
assert baskets.index.equals(invoice_ids) and baskets.dtypes.eq(bool).all()
display(pd.Series({'source_lines':len(raw),'eligible_product_lines':len(eligible),
    'eligible_invoices':len(invoice_ids),'eligible_IDs':len(customer_ids),
    'catalog_products':len(catalog),'catalog_empty_invoices':int((~baskets.any(axis=1)).sum())},name='coverage'))
display(popularity.head(8).assign(description=lambda t:t.StockCode.map(labels)))
display(baskets.iloc[:5,:6])

In [ ]:
# TODO 3: Inspect the raw lines from one invoice and its binary representation.
example_invoice = baskets.sum(axis=1).idxmax()
display(eligible[eligible.InvoiceNo.eq(example_invoice)]
        [['InvoiceNo','StockCode','Description','Quantity']].head(8))
print('Present catalog codes:',list(baskets.columns[baskets.loc[example_invoice]]))

# A tiny support lattice: support can only stay equal or fall when items are added.
toy_counts = []
for length in [1,2,3]:
    for itemset in combinations(['milk','bread','apples'],length):
        toy_counts.append({'itemset':' + '.join(itemset),
                           'count':int(toy[list(itemset)].all(axis=1).sum())})
display(pd.DataFrame(toy_counts))
print('At minimum count 2, any superset containing apples can be pruned.')

In [ ]:
RULE_COLUMNS = ['A','B','A_label','B_label','baskets','A_count','B_count',
                'joint_count','support','confidence','B_baseline','lift']

def mine_pair_rules(presence, min_joint_count=12):
    if not isinstance(min_joint_count,int) or isinstance(min_joint_count,bool) or min_joint_count<1:
        raise ValueError('Minimum joint count must be a positive integer.')
    if len(presence)==0 or len(presence.columns)<2 or min_joint_count>len(presence):
        return pd.DataFrame(columns=RULE_COLUMNS)
    # nextafter prevents floating rounding from excluding an exact boundary count.
    threshold = np.nextafter(min_joint_count/len(presence),0.0)
    itemsets = fpgrowth(presence,min_support=threshold,use_colnames=True,max_len=2)
    if itemsets.empty or not itemsets.itemsets.map(len).eq(2).any():
        return pd.DataFrame(columns=RULE_COLUMNS)
    generated = association_rules(itemsets,num_itemsets=len(presence),metric='confidence',min_threshold=0)
    rows = []
    for _,rule in generated.iterrows():
        A,B = next(iter(rule.antecedents)),next(iter(rule.consequents))
        rows.append({'A':A,'B':B,'A_label':labels.get(A,A),'B_label':labels.get(B,B),
            'baskets':len(presence),'A_count':int(round(rule['antecedent support']*len(presence))),
            'B_count':int(round(rule['consequent support']*len(presence))),
            'joint_count':int(round(rule.support*len(presence))),
            'support':rule.support,'confidence':rule.confidence,
            'B_baseline':rule['consequent support'],'lift':rule.lift})
    result = pd.DataFrame(rows,columns=RULE_COLUMNS)
    return result[result.joint_count.ge(min_joint_count)].sort_values(['A','B']).reset_index(drop=True)

minimum_count = 12
rules = mine_pair_rules(baskets,minimum_count)
print('Directed pair rules:',len(rules),'| Minimum joint count:',minimum_count,
      '| Baskets:',len(baskets))
display(rules[['A','B','joint_count','support','confidence','B_baseline','lift']].head(6).round(3))

## Investigation 1 — Popular consequence or informative combination?

**40 minutes; choose a route after recording your prediction.** A colleague wants three product combinations worth examining. Your task is to explain which evidence makes them candidates, not to recommend an immediate campaign.

Route A, **confidence versus lift**, compares the three highest-confidence rules with the three highest-lift rules under the same population, catalog, and minimum count. Inspect each consequent's baseline prevalence and the actual joint count. High confidence can reflect a common consequent; high lift can reflect a comparatively uncommon pair supported by limited evidence. Neither ordering is a business-value ranking. Explain one rule selected by each criterion, including the overlap if the same rules appear in both lists.

Route B, **follow one product family**, chooses one antecedent automatically from the mined results, displays its strongest candidate consequences, and follows one rule back to actual invoices. Compare descriptions and codes. Products may be color or size variants, packaging companions, or items purchased in broad wholesale orders; names suggest hypotheses, not verified intentions. You can change the selected antecedent to another displayed code after inspecting the baseline.

Both routes use a 2×2 count table for one selected rule: A and B, A without B, B without A, and neither. Hand-check the displayed support, confidence, and lift. Inspect an invoice containing both products and, when available, one containing A without B. The counterexample prevents “often together” from quietly becoming “always together.” Displayed quantities provide source context but do not change the binary denominators.

**Finished when:** explain three selected rules in plain English with joint count/N, A count, B baseline, confidence, and lift. State what differs when you reverse one arrow. Include a source example, a counterexample or explicit absence of one, and one reason to postpone an intervention. If no rules survive your settings, that is a valid result: report the constraints and return to the default settings before source inspection.

<details><summary>Hint if stuck</summary>

Read “A appears on … invoices; among those invoices, … also contain B; B appears on … of all invoices.” Then compare the two proportions. Use the prepared default pair before changing codes. You do not need to implement a mining algorithm or manually prepare a dataset.

</details>

In [ ]:
# TODO 4: Choose 'rankings' or 'product_family'. The underlying rules stay fixed.
investigation_route = 'rankings'
confidence_list = rules.sort_values(['confidence','joint_count','A','B'],ascending=[False,False,True,True]).head(3)
lift_list = rules.sort_values(['lift','joint_count','A','B'],ascending=[False,False,True,True]).head(3)
if investigation_route == 'rankings':
    print('Highest confidence:'); display(confidence_list.round(3))
    print('Highest lift:'); display(lift_list.round(3))
    shortlist = pd.concat([confidence_list,lift_list]).drop_duplicates(['A','B']).head(3)
elif investigation_route == 'product_family':
    focus_product = lift_list.iloc[0].A if len(lift_list) else None
    print('Focus product:',focus_product,labels.get(focus_product,''))
    shortlist = rules[rules.A.eq(focus_product)].sort_values(
        ['lift','joint_count','B'],ascending=[False,False,True]).head(3)
    display(shortlist.round(3))
else:
    raise ValueError('Choose rankings or product_family.')
if rules.empty:
    selected_A,selected_B = catalog[:2]
    print('No surviving rules. Inspect the default catalog pair as a no-rule example.')
else:
    selected_A,selected_B = lift_list.iloc[0][['A','B']]
# TODO 5: Replace selected_A/B with another pair from the displayed tables if desired.
evidence = rule_evidence(baskets,{selected_A},{selected_B})
display(evidence.round(3))
display(pd.DataFrame({'A → B':evidence,
                     'B → A':rule_evidence(baskets,{selected_B},{selected_A})}).round(3))

In [ ]:
in_A,in_B = baskets[selected_A],baskets[selected_B]
counts = pd.DataFrame([[int((in_A & in_B).sum()),int((in_A & ~in_B).sum())],
                      [int((~in_A & in_B).sum()),int((~in_A & ~in_B).sum())]],
                     index=['A present','A absent'],columns=['B present','B absent'])
display(counts)
print('A:',selected_A,labels.get(selected_A,''),'| B:',selected_B,labels.get(selected_B,''))
for name,mask in [('Both present',in_A & in_B),('A without B',in_A & ~in_B)]:
    matching = baskets.index[mask]
    if len(matching):
        invoice = matching[0]
        print(name,'— invoice',invoice)
        display(eligible[eligible.InvoiceNo.eq(invoice) & eligible.StockCode.isin([selected_A,selected_B])]
                [['InvoiceNo','StockCode','Description','Quantity']])
    else:
        print(name,'— no example in this population; do not invent one.')
fig,ax = plt.subplots(figsize=(7,3.5))
ax.bar(['B among A invoices','B among all invoices'],
       [evidence.confidence,evidence.B_baseline],color=['#277da8','#8ca6b8'])
ax.set(ylim=(0,1),ylabel='Observed fraction',title='Confidence needs the consequent baseline')
plt.tight_layout(); plt.show()

## Reading C — A shortlist is a hypothesis, not a deployment decision

**Read independently for 10–15 minutes, including the checkpoint.** Separate a descriptive pattern, a search setting, a validation claim, and a recommendation.

### The most striking number may not answer the question

Suppose two products occur together twice in a thousand baskets and each appears only twice. Their confidence is one in both directions and their lift is 500. The arithmetic is correct, but the evidence is two observations. A small change could make the apparent regularity disappear. A much more common pair may have lower lift and more evidence. Counts and proportions answer different questions; a responsible shortlist keeps both rather than sorting one score and calling the top result “best.”

Conversely, a product present in almost every basket can produce high confidence for many antecedents without adding much information. The eight-basket example gave 75% confidence and lift one. In real data, a high-lift pair can still be commercially trivial, already explained by product variants, or dominated by a few customers. Our data contain no margins, intervention costs, stock availability, or randomized campaign outcomes. Measures of co-occurrence do not supply those missing quantities.

A rule's direction changes its conditional question. It does not establish that the left product was bought first, prompted the other purchase, or should be advertised first. Both products may be recorded on one invoice without within-invoice timestamps. A claim about “next purchase” would require explicitly ordered events and an appropriate evaluation, not a new label for the same basket table. A plausible product recommendation is a hypothesis to test, not an experimentally demonstrated sales effect.

### Keep the population and the search separate

When minimum joint count increases, fewer combinations qualify. The surviving rules' counts and measures do not change when the presence table is fixed; the filter only changes which rules are shown. Lowering the threshold expands discovery but does not strengthen existing evidence. It can bring fragile low-count rules into view. If a filter yields no results, do not silently relax it until something attractive appears; record the empty result and explain any subsequent exploratory change.

Changing the basket unit is different. Merging all invoices for a customer creates a question about co-occurrence anywhere in that customer's observed history. There are fewer baskets, different marginal prevalences, and opportunities for products bought months apart to appear together. Neither representation is intrinsically wrong, but their confidence and lift do not describe the same event. A change in score does not establish an algorithmic improvement when the question itself changed.

Dropping all-false catalog rows is another population change. It conditions the analysis on buying at least one selected product. It raises the support fractions of retained combinations, while their conditional confidence can stay unchanged. Lift also changes because the consequent baseline changes. That is why our matrix construction explicitly reindexes to the full eligible invoice universe. What looks like a convenient removal of empty data can instead rewrite the evidence contract.

### Inspection needs supporting and contrary cases

A list of matching invoices helps establish that a combination is genuinely present under the chosen definition, but it does not explain why. Inspect invoices containing A without B as well, and consider whether repeated purchasing by a single account drives the finding. A customer can contribute several baskets, so the invoice count should not be described as an equivalent number of independent shoppers. The available extract also reflects selection of known IDs, an incomplete final month, and one historical retailer including wholesale purchases.

Descriptions help you propose a reason to investigate, such as matching colors or complementary packaging. They do not prove that customers use the products together. A useful recommendation identifies the next evidence needed: review a small group of source invoices, check product categorization, ask whether the pair is already merchandised together, or plan a later test with a relevant outcome. It can conclude that no rule yet deserves action.

### Discovery now, validation next

Today we selected the catalog, count threshold, and shortlist using one dataset. Reporting the same dataset's measures is descriptive exploration. It is not held-out performance. Searching many candidate rules gives many opportunities to find a striking number by chance; a high lift alone is not a statistical significance test. Selecting a few rules after seeing all their scores also means their observed strength should not be assumed to persist unchanged.

Week 5 will separate discovery from validation: choose the representation and rules on one part, freeze the shortlist, then evaluate that same list on another part, including rules with zero or undefined evidence. A temporal split may be appropriate for a question about later observations, but repeated customers, changing product availability, and source-window limits still need interpretation. We do not claim future reliability from today's threshold experiment or call a freshly mined list in each split validation of the original rules.

### Checkpoint

Write three sentences: what changing a count threshold tests; why customer-history baskets answer a different question from invoices; and why checking a rule on the data that selected it is not held-out evaluation. Add one fact that would be needed before recommending a sales intervention.

<details><summary>Self-check</summary>

A threshold change tests the shortlist's sensitivity to a selection boundary, with the underlying counts unchanged. Customer baskets span histories rather than single invoices. Reusing discovery data describes the selected observations rather than testing a frozen claim on new evidence. An intervention needs outcome/cost information and an appropriate comparison, not merely co-purchase lift.

</details>

## Investigation 2 — Change the question or change the filter?

**40 minutes; choose Route A or B and predict the result first.** Keep the catalog fixed so the effect of the intended change is identifiable. Every route ends with an evidence table: claim, original counts, controlled change, observed consequence, revised claim, and missing evidence.

Route A, **invoice versus customer history**, rebuilds the presence table with one row per eligible customer ID while keeping the same products and eligible source lines. Inspect the same selected A → B rule in both tables, even if it would fail a mining threshold in one representation. Trace a customer whose history contains both products but whose invoices never contain both, when such a case exists. Do not invent one if the selected pair has none; try another displayed pair or explain the absence. Counts from the two populations should not be presented as directly comparable sample sizes for the same question.

Route B, **support boundary**, compares minimum joint counts 12, 24, and 48 on exactly the same invoice matrix. Count surviving rules, inspect one lost rule where available, and verify that a surviving rule's metrics stay unchanged. An empty high-threshold list is a legitimate outcome. Then explain whether an excluded rule is unsupported under the stated threshold or proven uninteresting; those statements are not equivalent.

Everyone checks the denominator demonstration: remove only all-false catalog rows and compare the same fixed rule. Name the new conditioning population explicitly. Notice which counts stay constant, which proportions change, and why the apparent convenience of dropping empty rows is not a neutral cleanup.

**Finished when:** state which setting changed, inspect a concrete affected case, report the denominator, and write a revised recommendation. Use one or two evidence-table rows rather than a collection of unexplained plots. If you finish early, compare a second product pair or the alternative route; do not start an unbounded parameter search.

<details><summary>Hint if stuck</summary>

For Route A, read the `baskets`, `A_count`, and `joint_count` columns before the scores. For Route B, compare the row identified by the same A/B codes, not row numbers after sorting. Explain “not selected” separately from “did not occur.”

</details>

In [ ]:
# TODO 6: Choose 'basket_unit' or 'support_boundary'. No new catalog is selected.
challenge_route = 'basket_unit'
if challenge_route == 'basket_unit':
    customer_baskets = make_presence(eligible,'CustomerID',catalog,customer_ids)
    display(pd.DataFrame({'invoice':rule_evidence(baskets,{selected_A},{selected_B}),
        'customer_history':rule_evidence(customer_baskets,{selected_A},{selected_B})}).round(3))
    invoices_with_both = set(eligible.loc[eligible.InvoiceNo.isin(baskets.index[in_A & in_B]),'CustomerID'])
    histories_with_both = customer_baskets.index[customer_baskets[selected_A] & customer_baskets[selected_B]]
    cross_invoice_only = [identifier for identifier in histories_with_both if identifier not in invoices_with_both]
    print('IDs with both across history but never together on an eligible invoice:',len(cross_invoice_only))
    if cross_invoice_only:
        chosen_ID = cross_invoice_only[0]
        display(eligible[eligible.CustomerID.eq(chosen_ID) & eligible.StockCode.isin([selected_A,selected_B])]
            [['CustomerID','InvoiceNo','InvoiceDate','StockCode','Description']].head(12))
else:
    if challenge_route != 'support_boundary': raise ValueError('Choose basket_unit or support_boundary.')
    by_threshold = {count:mine_pair_rules(baskets,count) for count in [12,24,48]}
    display(pd.DataFrame([{'minimum_joint_count':count,'directed_rules':len(table)}
                          for count,table in by_threshold.items()]))
    comparison = []
    for count,table in by_threshold.items():
        row = table[table.A.eq(selected_A) & table.B.eq(selected_B)]
        comparison.append({'minimum_count':count,'selected':not row.empty,
            **rule_evidence(baskets,{selected_A},{selected_B}).to_dict()})
    display(pd.DataFrame(comparison).round(3))
    lost = by_threshold[12][~by_threshold[12].set_index(['A','B']).index.isin(
        by_threshold[48].set_index(['A','B']).index)]
    display(lost[['A','B','joint_count','confidence','lift']].head(4).round(3))

In [ ]:
# TODO 7: Explain the changed denominator without calling it a performance gain.
catalog_nonempty = baskets[baskets.any(axis=1)]
denominator_comparison = pd.DataFrame({
    'all_eligible_invoices':rule_evidence(baskets,{selected_A},{selected_B}),
    'only_catalog_nonempty':rule_evidence(catalog_nonempty,{selected_A},{selected_B})})
display(denominator_comparison.round(3))
# Both are computable, but the second conditions on buying a selected-catalog product.

## Synthesis — recommend an investigation, not an unsupported intervention

Use 15 minutes to write a 150–250-word memo with three rule explanations, one source/counterexample inspection, one controlled challenge, and a qualified next step. Each explanation needs counts, a denominator, the consequent baseline, and a statement of what the rule does not establish. Finish by revisiting your opening prediction about “every A buyer also buys B.”

**TODO 8:** Complete an evidence table with columns `claim`, `supporting_counts`, `challenge`, `observed_change`, `revised_claim`, and `next_evidence`. One well-supported recommendation to postpone action is preferable to a claim that a high lift demonstrates a profitable intervention.

A defensible sentence would be: “Under our invoice definition and selected catalog, these products co-occur more often than their marginal prevalences suggest; source inspection supports a product-family hypothesis, which needs independent validation.” Do not copy that conclusion if your measures do not support it. Report an inconclusive or ordinary-baseline result honestly.

### Practice and assessment

This notebook is classroom practice, not a new assignment or an announcement of a submission deadline. The assignment set is being reviewed separately; use only the instructor's explicitly released briefs for assessment requirements. Earlier draft proposals are not new obligations introduced by this lecture.

### Next lecture

Week 5 asks which discovered rules survive a fresh comparison. We will freeze a shortlist, examine held-out evidence, and consider redundancy, rare-item effects, and sensitivity. Today's counts are the foundation; they are not already that validation.

## Reading path — visualize the denominator, then inspect the mining method

All essential explanations are in this notebook. These optional readings require no account or dataset upload.

1. **Start visually:** [Victor Powell / Setosa — Conditional probability explained visually](https://setosa.io/conditional/), 8–10 minutes of exploration. Change the event probabilities and overlap, then switch between the two conditional perspectives and the world view. Which denominator changes when you reverse the conditioning event? This supports confidence intuition; the animation is not a shopping dataset or a demonstration of causal effects.
2. **Inspect the method:** [mlxtend — FP-growth illustrated guide](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/fpgrowth/), 10–15 minutes focused on transaction encoding and the explanation of frequent-pattern growth. Connect the example's boolean columns, support threshold, and length limit to our catalog and invoice matrix. The library performs mining; our counts and inspection establish what the output means.
3. **Check the measures:** [mlxtend — Association rules guide](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/), approximately 10 minutes focused on support, confidence, and lift. Compare the examples' antecedent and consequent supports with our four-cell table. Other metrics are optional reference, not additional terms to memorize today.

Read the visual explanation before technical documentation if you want a refresher on conditioning. Avoid interpreting the animation's sequence of falling balls as a temporal meaning for a basket-rule arrow: the arrow in our analysis only defines a conditional co-occurrence.

## Optional extension — check the search, not a speed claim

Compare Apriori and FP-growth on only the first ten catalog products, with the same invoice rows, minimum support, and maximum length two. They should return the same qualifying itemsets, apart from representation or ordering differences. This is a correctness check on a small bounded case, not evidence that one algorithm is universally faster. Do not remove limits to make the runtime comparison more dramatic.

For a second optional question, investigate whether a shortlisted rule is concentrated in a few customer histories. Count distinct contributing IDs and distinguish that denominator from invoice count. Keep the comparison exploratory and do not manufacture independent customer labels from transaction rows.

In [ ]:
small = baskets.iloc[:,:10]
threshold = np.nextafter(minimum_count/len(small),0.0)
fp_small = fpgrowth(small,min_support=threshold,use_colnames=True,max_len=2)
ap_small = apriori(small,min_support=threshold,use_colnames=True,max_len=2)
fp_map = {tuple(sorted(row.itemsets)):row.support for _,row in fp_small.iterrows()}
ap_map = {tuple(sorted(row.itemsets)):row.support for _,row in ap_small.iterrows()}
assert fp_map.keys()==ap_map.keys()
assert all(np.isclose(fp_map[key],ap_map[key]) for key in fp_map)
print('Same bounded frequent itemsets from both algorithms:',len(fp_map))